# 06 検証：参照の結果と照合する
今回作った結果表を、参照の結果表とセルごとに比べます。同じコード・同じデータ・同じ乱数なら、数字は一致します。

- **デモ**：参照は `reference/demo_tables`（公開時に同じコードで合成データから作ったもの）。一致すれば、環境が正しく作れています。
- **実データ**：参照は `pbrtqc.local.json` の `reference_tables`（以前の解析の結果表）。一致すれば、論文の数字を独立に再現できたことになります。

照合結果は `work/verify_report.csv`（デモは `work_demo/`）に保存されます。中身は集計表の一致・不一致だけで、患者データは含みません。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "pbrtqc").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 200)
from pbrtqc import config as C, pipeline
print(C.describe())

> **いまのモード**は上のセルの `profile=` で確認してください（`demo` = 合成データ、`real` = 実データ）。モードを変えたら、VS Code の「Restart」でカーネルを再起動してから上から実行し直します。

In [ ]:
from pbrtqc import verify
rep = verify.compare()
print(f"{(rep['判定'] == '一致').sum()} / {len(rep)} 件が一致")
rep

## 一致しなかった表があるとき
`内容` 列に、列・行数・値のどれが違うかが出ます。版の違い（00 で確認）、途中で止まったステップ（`run_log_*.json`）を先に確認してください。

In [ ]:
rep[rep["判定"] != "一致"]

## 実行の記録（コードの版、パッケージの版、各ステップの時間、入力ファイルの SHA-256）

In [ ]:
for p in sorted(C.WORK.glob("run_log*.json")):
    lg = json.loads(p.read_text(encoding="utf-8"))
    print(p.name, "|", lg["environment"]["code_version"], lg["environment"]["packages"], "|", lg.get("started"), "→", lg.get("finished"))
    print("   steps:", ", ".join(f"{s['step']}({s['seconds']}s, exit {s['returncode']})" for s in lg["steps"]))